# Bayesian A/B Testing for Subscription Conversion and Revenue

Imagine we are testing whether a new onboarding or checkout experience improves early monetization.

This notebook walks through a lightweight Bayesian A/B testing workflow using subscription users and orders:

| Section | What we do |
|---|---|
| Setup | Load users and orders, inspect data coverage |
| Assignment | Randomly assign users to control and treatment for illustration |
| Outcomes | Build user-level conversion, revenue, and churn outcomes after signup |
| Sanity checks | Check assignment balance and observed outcomes |
| Frequentist baseline | Run a short observed-rate comparison |
| Bayesian conversion model | Use a Beta-Binomial model for 30-day conversion |
| Decision framework | Turn posterior probabilities into a practical ship/no-ship call |
| Synthetic effect | Inject a small treatment effect to show a winning experiment |
| Revenue model | Estimate revenue per user when revenue exists |
| Prior sensitivity | Compare weak and stronger priors |
| Executive summary | Summarize what the analysis answers and what it does not |

**Important caveat:** `variant` is assigned after the fact in this notebook. That makes this an illustration of Bayesian A/B testing mechanics, not evidence of a real causal product treatment. A real causal read requires treatment assignment to happen before user outcomes are generated.

## 1. Setup

We use only lightweight scientific Python: `pandas`, `numpy`, `scipy`, and `matplotlib`. The notebook is designed to run from the repo root or from the `notebooks/` directory.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
from scipy import stats

SEED = 42
rng = np.random.default_rng(SEED)

# Works whether the notebook is launched from the repo root or from notebooks/.
DATA_DIR = Path("data") / "synthetic"
if not DATA_DIR.exists():
    DATA_DIR = Path("..") / "data" / "synthetic"

N_POSTERIOR_SAMPLES = 100_000
N_REVENUE_SAMPLES = 50_000
MINIMUM_PRACTICAL_RELATIVE_LIFT = 0.02
CREDIBLE_MASS = 0.94

plt.rcParams.update(
    {
        "figure.figsize": (9, 4.8),
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.titleweight": "bold",
        "axes.labelsize": 10,
        "axes.titlesize": 12,
        "legend.frameon": False,
    }
)

print(f"DATA_DIR: {DATA_DIR.resolve()}")

In [ ]:
users = pd.read_csv(DATA_DIR / "users.csv", parse_dates=["signup_at", "churn_at"])
orders = pd.read_csv(
    DATA_DIR / "orders.csv",
    parse_dates=["billed_at", "period_start", "period_end"],
)

print(f"Users rows:  {len(users):,}")
print(f"Orders rows: {len(orders):,}")
print(f"Users signup range: {users['signup_at'].min().date()} to {users['signup_at'].max().date()}")
print(f"Orders billed range: {orders['billed_at'].min().date()} to {orders['billed_at'].max().date()}")

print("\nUser columns:")
print(sorted(users.columns.tolist()))
print("\nOrder columns:")
print(sorted(orders.columns.tolist()))

## Reusable functions

The core logic is wrapped in small functions so the notebook reads like an analysis but can still be reused. These are intentionally plain Python functions rather than a package dependency, so a reader can understand the full workflow top to bottom.

In [ ]:
COMMON_REVENUE_COLUMNS = [
    "amount",
    "amount_usd",
    "revenue",
    "total",
    "price",
    "net_revenue",
]


def detect_revenue_column(orders: pd.DataFrame) -> str | None:
    """Return the first common revenue column found in the orders table."""
    normalized = {col.lower(): col for col in orders.columns}
    for candidate in COMMON_REVENUE_COLUMNS:
        if candidate in normalized:
            return normalized[candidate]
    return None


def hdi_interval(values: np.ndarray, mass: float = 0.94) -> tuple[float, float]:
    """Compute the narrowest interval containing the requested posterior mass."""
    values = np.sort(np.asarray(values, dtype=float))
    if len(values) == 0:
        return (np.nan, np.nan)
    interval_idx = int(np.floor(mass * len(values)))
    if interval_idx < 1 or interval_idx >= len(values):
        return (float(values[0]), float(values[-1]))
    widths = values[interval_idx:] - values[: len(values) - interval_idx]
    start = int(np.argmin(widths))
    return (float(values[start]), float(values[start + interval_idx]))


def pct(value: float) -> str:
    """Format a decimal as a percentage."""
    if pd.isna(value):
        return "n/a"
    return f"{value:.1%}"


def money(value: float) -> str:
    """Format a number as dollars."""
    if pd.isna(value):
        return "n/a"
    return f"${value:,.2f}"

In [ ]:
def make_user_level_outcomes(
    users: pd.DataFrame,
    orders: pd.DataFrame,
    revenue_col: str | None = None,
) -> tuple[pd.DataFrame, str | None]:
    """Create user-level outcomes measured after signup.

    Outcomes:
    - converted_14d: at least one order in the first 14 days after signup.
    - converted_30d: at least one order in the first 30 days after signup.
    - revenue_30d: total revenue in the first 30 days after signup, when revenue exists.
    - has_revenue_30d: revenue_30d > 0, when revenue exists.
    - churned_30d: churn observed in the first 30 days after signup, when churn_at exists.
    """
    required_user_cols = {"user_id", "signup_at"}
    required_order_cols = {"user_id", "billed_at"}
    missing_user_cols = required_user_cols - set(users.columns)
    missing_order_cols = required_order_cols - set(orders.columns)
    if missing_user_cols:
        raise ValueError(f"users is missing required columns: {sorted(missing_user_cols)}")
    if missing_order_cols:
        raise ValueError(f"orders is missing required columns: {sorted(missing_order_cols)}")

    detected_revenue_col = revenue_col or detect_revenue_column(orders)
    user_outcomes = users[["user_id", "signup_at"]].copy()

    joined_orders = orders.merge(
        users[["user_id", "signup_at"]],
        on="user_id",
        how="inner",
        validate="many_to_one",
    )
    joined_orders = joined_orders[joined_orders["billed_at"] >= joined_orders["signup_at"]].copy()

    within_14d = joined_orders["billed_at"] < joined_orders["signup_at"] + pd.Timedelta(days=14)
    within_30d = joined_orders["billed_at"] < joined_orders["signup_at"] + pd.Timedelta(days=30)

    converted_14d_users = set(joined_orders.loc[within_14d, "user_id"])
    converted_30d_users = set(joined_orders.loc[within_30d, "user_id"])

    user_outcomes["converted_14d"] = user_outcomes["user_id"].isin(converted_14d_users)
    user_outcomes["converted_30d"] = user_outcomes["user_id"].isin(converted_30d_users)

    if detected_revenue_col is None:
        print(
            "No revenue column found. Checked common names: "
            f"{', '.join(COMMON_REVENUE_COLUMNS)}. Revenue analysis will be skipped."
        )
    else:
        joined_orders[detected_revenue_col] = pd.to_numeric(
            joined_orders[detected_revenue_col],
            errors="coerce",
        ).fillna(0.0)
        revenue_30d = (
            joined_orders.loc[within_30d]
            .groupby("user_id")[detected_revenue_col]
            .sum()
            .rename("revenue_30d")
        )
        user_outcomes = user_outcomes.merge(
            revenue_30d,
            on="user_id",
            how="left",
        )
        user_outcomes["revenue_30d"] = user_outcomes["revenue_30d"].fillna(0.0)
        user_outcomes["has_revenue_30d"] = user_outcomes["revenue_30d"] > 0
        print(f"Detected revenue column: {detected_revenue_col}")

    if "churn_at" in users.columns:
        churn = users[["user_id", "signup_at", "churn_at"]].copy()
        churn["churned_30d"] = (
            churn["churn_at"].notna()
            & (churn["churn_at"] >= churn["signup_at"])
            & (churn["churn_at"] < churn["signup_at"] + pd.Timedelta(days=30))
        )
        user_outcomes = user_outcomes.merge(
            churn[["user_id", "churned_30d"]],
            on="user_id",
            how="left",
        )
    else:
        print("No churn_at column found. churned_30d guardrail will be skipped.")

    return user_outcomes.drop(columns=["signup_at"]), detected_revenue_col


def summarize_experiment(df: pd.DataFrame, outcome_col: str) -> pd.DataFrame:
    """Summarize sample size, conversion, revenue, and optional churn by variant."""
    summary = (
        df.groupby("variant")
        .agg(
            users=("user_id", "size"),
            conversions=(outcome_col, "sum"),
            conversion_rate=(outcome_col, "mean"),
        )
        .reindex(["control", "treatment"])
    )
    summary["conversions"] = summary["conversions"].astype(int)

    if "revenue_30d" in df.columns:
        revenue_summary = df.groupby("variant")["revenue_30d"].agg(
            mean_revenue_per_user="mean",
            median_revenue_per_user="median",
        )
        summary = summary.join(revenue_summary)

    if "churned_30d" in df.columns:
        summary = summary.join(df.groupby("variant")["churned_30d"].mean().rename("churned_30d_rate"))

    return summary


def balance_table(df: pd.DataFrame, column: str) -> pd.DataFrame:
    """Return row-normalized variant distribution for a baseline column."""
    table = pd.crosstab(df[column], df["variant"], normalize="columns")
    return table.reindex(columns=["control", "treatment"]).fillna(0.0)

In [ ]:
def bayesian_beta_binomial_ab_test(
    df: pd.DataFrame,
    outcome_col: str,
    variant_col: str = "variant",
    control_label: str = "control",
    treatment_label: str = "treatment",
    alpha: float = 1,
    beta: float = 1,
    n_samples: int = 100_000,
    seed: int = 42,
) -> dict:
    """Run a conjugate Bayesian A/B test for a binary outcome."""
    if outcome_col not in df.columns:
        raise ValueError(f"{outcome_col} is not in df")

    analysis_df = df[df[variant_col].isin([control_label, treatment_label])].copy()
    analysis_df[outcome_col] = analysis_df[outcome_col].astype(bool)

    counts = {}
    for label in [control_label, treatment_label]:
        arm = analysis_df[analysis_df[variant_col] == label]
        successes = int(arm[outcome_col].sum())
        n = int(len(arm))
        counts[label] = {
            "n": n,
            "successes": successes,
            "failures": n - successes,
            "observed_rate": successes / n if n else np.nan,
        }

    local_rng = np.random.default_rng(seed)
    control_samples = local_rng.beta(
        alpha + counts[control_label]["successes"],
        beta + counts[control_label]["failures"],
        size=n_samples,
    )
    treatment_samples = local_rng.beta(
        alpha + counts[treatment_label]["successes"],
        beta + counts[treatment_label]["failures"],
        size=n_samples,
    )
    absolute_lift = treatment_samples - control_samples
    relative_lift = treatment_samples / np.maximum(control_samples, 1e-12) - 1

    return {
        "outcome_col": outcome_col,
        "control_label": control_label,
        "treatment_label": treatment_label,
        "prior": {"alpha": alpha, "beta": beta},
        "counts": counts,
        "control": control_samples,
        "treatment": treatment_samples,
        "absolute_lift": absolute_lift,
        "relative_lift": relative_lift,
    }


def summarize_posterior_results(samples: dict) -> pd.DataFrame:
    """Summarize posterior conversion rates and lift."""
    rows = []
    for key, label in [
        ("control", "Control conversion rate"),
        ("treatment", "Treatment conversion rate"),
        ("absolute_lift", "Absolute lift"),
        ("relative_lift", "Relative lift"),
    ]:
        values = samples[key]
        low, high = hdi_interval(values, CREDIBLE_MASS)
        rows.append(
            {
                "metric": label,
                "posterior_mean": float(np.mean(values)),
                "hdi_low": low,
                "hdi_high": high,
            }
        )
    return pd.DataFrame(rows)


def plot_conversion_posteriors(samples: dict) -> None:
    """Plot posterior conversion distributions for control and treatment."""
    fig, ax = plt.subplots()
    for key, label, color in [
        ("control", "Control", "#4C78A8"),
        ("treatment", "Treatment", "#F58518"),
    ]:
        values = samples[key]
        ax.hist(values, bins=80, density=True, alpha=0.35, label=label, color=color)
        ax.axvline(np.mean(values), color=color, linewidth=2)
    ax.set_title("Posterior Conversion Rate by Variant")
    ax.set_xlabel("30-day conversion rate")
    ax.set_ylabel("Posterior density")
    ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.legend()
    plt.tight_layout()
    plt.show()


def plot_lift_distribution(samples: dict, lift_type: str = "relative") -> None:
    """Plot posterior lift distribution with zero and HDI markers."""
    if lift_type not in {"relative", "absolute"}:
        raise ValueError("lift_type must be 'relative' or 'absolute'")

    values = samples["relative_lift"] if lift_type == "relative" else samples["absolute_lift"]
    scale = 100 if lift_type == "relative" else 1
    plotted = values * scale
    low, high = hdi_interval(values, CREDIBLE_MASS)
    mean_value = float(np.mean(values))
    prob_win = float(np.mean(samples["treatment"] > samples["control"]))

    fig, ax = plt.subplots()
    ax.hist(plotted, bins=90, density=True, color="#54A24B", alpha=0.60)
    ax.axvline(0, color="black", linewidth=1.5, label="No lift")
    ax.axvline(mean_value * scale, color="#E45756", linewidth=2, label="Posterior mean")
    ax.axvspan(low * scale, high * scale, color="#72B7B2", alpha=0.20, label=f"{int(CREDIBLE_MASS * 100)}% HDI")

    if lift_type == "relative":
        ax.set_xlabel("Relative lift (%)")
        summary_text = (
            f"P(treatment > control): {prob_win:.1%}\n"
            f"Expected relative lift: {mean_value:.1%}\n"
            f"{int(CREDIBLE_MASS * 100)}% HDI: {low:.1%} to {high:.1%}"
        )
    else:
        ax.set_xlabel("Absolute lift, percentage points")
        summary_text = (
            f"P(treatment > control): {prob_win:.1%}\n"
            f"Expected absolute lift: {mean_value:.2%}\n"
            f"{int(CREDIBLE_MASS * 100)}% HDI: {low:.2%} to {high:.2%}"
        )
    ax.set_title(f"Posterior Distribution of {lift_type.title()} Lift")
    ax.set_ylabel("Posterior density")
    ax.text(0.98, 0.95, summary_text, transform=ax.transAxes, ha="right", va="top")
    ax.legend(loc="upper left")
    plt.tight_layout()
    plt.show()

## 2. Create experiment assignment

The assignment below follows the requested structure: users are split 50/50 into control and treatment with a reproducible random seed.

**Causal caveat:** this is simulated assignment after outcomes already exist in the synthetic dataset. It is useful for illustrating mechanics, but it should not be read as a causal estimate of a real onboarding or checkout change.

In [ ]:
experiment = users[["user_id", "signup_at", "acquisition_channel", "country"]].copy()
experiment["variant"] = np.where(
    rng.random(len(experiment)) < 0.5,
    "control",
    "treatment",
)
experiment["signup_month"] = experiment["signup_at"].dt.to_period("M").astype(str)

assignment_counts = experiment["variant"].value_counts().reindex(["control", "treatment"])
assignment_counts.to_frame("users")

## 3. Create outcome metrics

We now construct user-level outcomes after signup. The date logic is explicit:

- Count only orders with `billed_at >= signup_at`.
- For 14-day conversion, require `billed_at < signup_at + 14 days`.
- For 30-day conversion and revenue, require `billed_at < signup_at + 30 days`.

If a recognizable revenue column is missing, the notebook prints a message and skips the revenue model later.

In [ ]:
outcomes, revenue_col = make_user_level_outcomes(users, orders)
experiment = experiment.merge(outcomes, on="user_id", how="left", validate="one_to_one")

outcome_cols = [
    col
    for col in ["converted_14d", "converted_30d", "revenue_30d", "has_revenue_30d", "churned_30d"]
    if col in experiment.columns
]

print(f"Outcome columns created: {', '.join(outcome_cols)}")
experiment[["user_id", "variant", "converted_14d", "converted_30d"] + [col for col in ["revenue_30d", "churned_30d"] if col in experiment.columns]].head()

## 4. Sanity checks

Randomization balance matters because treatment and control should be comparable before the treatment begins. If one group has more high-intent acquisition channels, more high-income countries, or a different signup-time mix, the outcome comparison can be biased.

For a real online experiment, these checks should be run before reading outcome results.

In [ ]:
users_by_variant = experiment["variant"].value_counts().reindex(["control", "treatment"]).to_frame("users")
users_by_variant["share"] = users_by_variant["users"] / users_by_variant["users"].sum()
users_by_variant.style.format({"share": "{:.1%}"})

In [ ]:
print("Acquisition channel balance")
display(balance_table(experiment, "acquisition_channel").style.format("{:.1%}"))

print("Country balance")
display(balance_table(experiment, "country").style.format("{:.1%}"))

print("Signup month balance")
display(balance_table(experiment, "signup_month").style.format("{:.1%}"))

In [ ]:
outcome_summary = summarize_experiment(experiment, "converted_30d")
formatters = {
    "conversion_rate": "{:.2%}",
    "mean_revenue_per_user": "${:,.2f}",
    "median_revenue_per_user": "${:,.2f}",
    "churned_30d_rate": "{:.2%}",
}
outcome_summary.style.format({col: fmt for col, fmt in formatters.items() if col in outcome_summary.columns})

## 5. Frequentist quick comparison baseline

Before the Bayesian model, it is useful to compute the observed difference and a simple two-proportion z-test. This is just a baseline comparison; the rest of the notebook focuses on posterior probabilities and decision-making.

In [ ]:
control = experiment[experiment["variant"] == "control"]
treatment = experiment[experiment["variant"] == "treatment"]

control_n = len(control)
treatment_n = len(treatment)
control_conversions = int(control["converted_30d"].sum())
treatment_conversions = int(treatment["converted_30d"].sum())
control_rate = control_conversions / control_n
treatment_rate = treatment_conversions / treatment_n
observed_absolute_lift = treatment_rate - control_rate
observed_relative_lift = treatment_rate / control_rate - 1 if control_rate > 0 else np.nan

pooled_rate = (control_conversions + treatment_conversions) / (control_n + treatment_n)
standard_error = np.sqrt(pooled_rate * (1 - pooled_rate) * (1 / control_n + 1 / treatment_n))
z_score = observed_absolute_lift / standard_error if standard_error > 0 else np.nan
p_value = 2 * (1 - stats.norm.cdf(abs(z_score))) if pd.notna(z_score) else np.nan

frequentist_baseline = pd.DataFrame(
    {
        "metric": [
            "Control conversion rate",
            "Treatment conversion rate",
            "Observed absolute lift",
            "Observed relative lift",
            "Two-proportion z-test p-value",
        ],
        "value": [
            pct(control_rate),
            pct(treatment_rate),
            pct(observed_absolute_lift),
            pct(observed_relative_lift),
            f"{p_value:.4f}" if pd.notna(p_value) else "n/a",
        ],
    }
)
frequentist_baseline

## 6. Bayesian conversion model: Beta-Binomial

Each user either converts within 30 days or does not. For each variant, the true conversion rate is unknown.

Model:

$$theta_{control} \sim Beta(alpha, beta)$$

$$theta_{treatment} \sim Beta(alpha, beta)$$

$$conversions_{control} \sim Binomial(n_{control}, theta_{control})$$

$$conversions_{treatment} \sim Binomial(n_{treatment}, theta_{treatment})$$

We start with a weak prior, `Beta(1, 1)`, which is uniform over conversion rates. With a conjugate Beta-Binomial model, the posterior is available in closed form:

$$theta \mid data \sim Beta(alpha + conversions, beta + non\_conversions)$$

From posterior samples, we estimate:

- absolute lift: `theta_treatment - theta_control`
- relative lift: `theta_treatment / theta_control - 1`
- `P(theta_treatment > theta_control)`
- `P(relative_lift > 2%)`
- 94% highest density intervals for rates and lift

In [ ]:
conversion_samples = bayesian_beta_binomial_ab_test(
    experiment,
    outcome_col="converted_30d",
    alpha=1,
    beta=1,
    n_samples=N_POSTERIOR_SAMPLES,
    seed=SEED,
)

posterior_summary = summarize_posterior_results(conversion_samples)
posterior_summary_display = posterior_summary.copy()
posterior_summary_display["posterior_mean"] = posterior_summary_display.apply(
    lambda row: pct(row["posterior_mean"]), axis=1
)
posterior_summary_display["94% HDI"] = posterior_summary.apply(
    lambda row: f"{pct(row['hdi_low'])} to {pct(row['hdi_high'])}", axis=1
)
posterior_summary_display[["metric", "posterior_mean", "94% HDI"]]

In [ ]:
prob_treatment_wins = float(np.mean(conversion_samples["treatment"] > conversion_samples["control"]))
prob_beats_practical_lift = float(
    np.mean(conversion_samples["relative_lift"] > MINIMUM_PRACTICAL_RELATIVE_LIFT)
)
relative_lift_hdi = hdi_interval(conversion_samples["relative_lift"], CREDIBLE_MASS)
absolute_lift_hdi = hdi_interval(conversion_samples["absolute_lift"], CREDIBLE_MASS)

conversion_decision_metrics = pd.DataFrame(
    {
        "metric": [
            "P(treatment > control)",
            f"P(relative lift > {MINIMUM_PRACTICAL_RELATIVE_LIFT:.0%})",
            "Expected relative lift",
            "94% HDI relative lift",
            "Expected absolute lift",
            "94% HDI absolute lift",
        ],
        "value": [
            pct(prob_treatment_wins),
            pct(prob_beats_practical_lift),
            pct(np.mean(conversion_samples["relative_lift"])),
            f"{pct(relative_lift_hdi[0])} to {pct(relative_lift_hdi[1])}",
            pct(np.mean(conversion_samples["absolute_lift"])),
            f"{pct(absolute_lift_hdi[0])} to {pct(absolute_lift_hdi[1])}",
        ],
    }
)
conversion_decision_metrics

### Prior sensitivity preview

A weak `Beta(1, 1)` prior lets the data dominate. If the business has historical evidence that conversion is usually low, we can use priors such as `Beta(2, 8)` or `Beta(10, 90)`. A fuller prior sensitivity table appears later.

In [ ]:
preview_priors = [(1, 1), (2, 8)]
preview_rows = []
for prior_alpha, prior_beta in preview_priors:
    prior_samples = bayesian_beta_binomial_ab_test(
        experiment,
        outcome_col="converted_30d",
        alpha=prior_alpha,
        beta=prior_beta,
        n_samples=N_POSTERIOR_SAMPLES,
        seed=SEED,
    )
    preview_rows.append(
        {
            "prior": f"Beta({prior_alpha}, {prior_beta})",
            "P(treatment > control)": np.mean(prior_samples["treatment"] > prior_samples["control"]),
            "expected_relative_lift": np.mean(prior_samples["relative_lift"]),
        }
    )

pd.DataFrame(preview_rows).style.format(
    {
        "P(treatment > control)": "{:.1%}",
        "expected_relative_lift": "{:.1%}",
    }
)

## 7. Visualization

The charts below show the posterior distributions directly. The lift charts include a vertical line at zero, the posterior mean, and the 94% HDI.

In [ ]:
plot_conversion_posteriors(conversion_samples)

In [ ]:
plot_lift_distribution(conversion_samples, lift_type="absolute")

In [ ]:
plot_lift_distribution(conversion_samples, lift_type="relative")

## 8. Decision framework

A Bayesian test should end with a decision rule, not just a chart.

Example rule:

Ship treatment if all are true:

1. `P(treatment > control) >= 0.95`
2. `P(relative_lift > minimum_practical_lift) >= 0.80`
3. No guardrail metric is harmed

Otherwise, continue the experiment, keep control, or collect more data.

Because assignment here is simulated after the fact, the recommendation below is a demonstration of the decision framework rather than a real product launch recommendation.

In [ ]:
ship_probability_ok = prob_treatment_wins >= 0.95
practical_lift_ok = prob_beats_practical_lift >= 0.80

if "churned_30d" in experiment.columns:
    churn_rates = experiment.groupby("variant")["churned_30d"].mean().reindex(["control", "treatment"])
    churn_delta = churn_rates.loc["treatment"] - churn_rates.loc["control"]
    guardrail_ok = churn_delta <= 0
    guardrail_interpretation = (
        "Treatment churn is not higher than control"
        if guardrail_ok
        else "Treatment churn is higher than control"
    )
    guardrail_value = pct(churn_delta)
else:
    guardrail_ok = True
    guardrail_interpretation = "No churn guardrail available"
    guardrail_value = "n/a"

if ship_probability_ok and practical_lift_ok and guardrail_ok:
    decision = "Ship treatment under this illustrative rule"
else:
    decision = "Do not ship yet under this illustrative rule"

decision_table = pd.DataFrame(
    {
        "metric": [
            "P(treatment > control)",
            f"P(relative lift > {MINIMUM_PRACTICAL_RELATIVE_LIFT:.0%})",
            "Guardrail: treatment minus control churned_30d",
            "Decision",
            "Causal caveat",
        ],
        "value": [
            pct(prob_treatment_wins),
            pct(prob_beats_practical_lift),
            guardrail_value,
            decision,
            "Assignment is simulated after outcomes; use as mechanics only",
        ],
        "interpretation": [
            "Pass" if ship_probability_ok else "Does not pass 95% threshold",
            "Pass" if practical_lift_ok else "Does not pass 80% practical-lift threshold",
            guardrail_interpretation,
            "Applies the pre-defined rule",
            "A real causal launch decision needs pre-outcome randomization",
        ],
    }
)
decision_table

## 9. Optional synthetic treatment effect section

The observed assignment above is simulated after the fact. To show what a winning experiment looks like, we can copy the experiment dataframe and inject a small synthetic lift into treatment conversion.

This is **simulation for learning only**. We flip a reproducible set of treatment non-converters to converters, then rerun the same Bayesian model.

In [ ]:
def inject_synthetic_treatment_effect(
    df: pd.DataFrame,
    outcome_col: str,
    absolute_lift_points: float = 0.03,
    seed: int = 42,
) -> tuple[pd.DataFrame, int]:
    """Flip treatment non-converters to converters to simulate a winning experiment."""
    simulated = df.copy()
    local_rng = np.random.default_rng(seed)

    treatment_mask = simulated["variant"].eq("treatment")
    treatment_n = int(treatment_mask.sum())
    current_successes = int(simulated.loc[treatment_mask, outcome_col].sum())
    target_successes = int(np.ceil(current_successes + absolute_lift_points * treatment_n))
    flips_needed = max(0, target_successes - current_successes)

    eligible = simulated.index[treatment_mask & ~simulated[outcome_col].astype(bool)].to_numpy()
    flips = min(flips_needed, len(eligible))
    if flips > 0:
        selected = local_rng.choice(eligible, size=flips, replace=False)
        simulated.loc[selected, outcome_col] = True
        if "has_revenue_30d" in simulated.columns:
            simulated.loc[selected, "has_revenue_30d"] = True
    return simulated, flips


simulated_experiment, flipped_users = inject_synthetic_treatment_effect(
    experiment,
    outcome_col="converted_30d",
    absolute_lift_points=0.03,
    seed=SEED + 7,
)

print(f"Synthetic treatment effect injected by flipping {flipped_users:,} treatment non-converters.")

simulated_samples = bayesian_beta_binomial_ab_test(
    simulated_experiment,
    outcome_col="converted_30d",
    alpha=1,
    beta=1,
    n_samples=N_POSTERIOR_SAMPLES,
    seed=SEED,
)

comparison = pd.DataFrame(
    {
        "scenario": ["Observed after-the-fact assignment", "Synthetic winning treatment"],
        "P(treatment > control)": [
            np.mean(conversion_samples["treatment"] > conversion_samples["control"]),
            np.mean(simulated_samples["treatment"] > simulated_samples["control"]),
        ],
        "expected_relative_lift": [
            np.mean(conversion_samples["relative_lift"]),
            np.mean(simulated_samples["relative_lift"]),
        ],
        "expected_absolute_lift": [
            np.mean(conversion_samples["absolute_lift"]),
            np.mean(simulated_samples["absolute_lift"]),
        ],
    }
)
comparison.style.format(
    {
        "P(treatment > control)": "{:.1%}",
        "expected_relative_lift": "{:.1%}",
        "expected_absolute_lift": "{:.2%}",
    }
)

In [ ]:
plot_lift_distribution(simulated_samples, lift_type="relative")

## 10. Bayesian revenue model, if revenue exists

Revenue is usually zero-inflated: many users pay nothing, and purchasers have a skewed distribution of order values. A pragmatic model separates revenue per user into two pieces:

1. Probability of purchase: Beta-Binomial model on `has_revenue_30d`.
2. Revenue among purchasers: Bayesian bootstrap over positive `revenue_30d` values.

For each posterior draw:

$$revenue\_per\_user = purchase\_rate \times revenue\_given\_purchase$$

If no revenue column exists, or if there are too few purchasers to estimate purchaser revenue, this section prints a clear skip message.

In [ ]:
def bayesian_bootstrap_mean(
    values: np.ndarray,
    n_samples: int,
    seed: int = 42,
    chunk_size: int = 2_000,
) -> np.ndarray:
    """Posterior bootstrap draws for the mean of a positive continuous metric."""
    values = np.asarray(values, dtype=float)
    if len(values) == 0:
        raise ValueError("Cannot bootstrap an empty array.")
    if len(values) == 1:
        return np.repeat(values[0], n_samples)

    local_rng = np.random.default_rng(seed)
    draws = []
    for start in range(0, n_samples, chunk_size):
        current_size = min(chunk_size, n_samples - start)
        weights = local_rng.exponential(1.0, size=(current_size, len(values)))
        weights /= weights.sum(axis=1, keepdims=True)
        draws.append(weights @ values)
    return np.concatenate(draws)


if revenue_col is None or "revenue_30d" not in experiment.columns:
    print("Skipping revenue model: no recognized revenue column exists in orders.csv.")
elif experiment["has_revenue_30d"].sum() < 10:
    print(
        "Skipping revenue model: fewer than 10 users have positive 30-day revenue, "
        "which is too sparse for a useful revenue walkthrough."
    )
else:
    purchase_samples = bayesian_beta_binomial_ab_test(
        experiment,
        outcome_col="has_revenue_30d",
        alpha=1,
        beta=1,
        n_samples=N_REVENUE_SAMPLES,
        seed=SEED,
    )

    positive_control_revenue = experiment.loc[
        (experiment["variant"] == "control") & (experiment["revenue_30d"] > 0),
        "revenue_30d",
    ].to_numpy()
    positive_treatment_revenue = experiment.loc[
        (experiment["variant"] == "treatment") & (experiment["revenue_30d"] > 0),
        "revenue_30d",
    ].to_numpy()

    if len(positive_control_revenue) < 5 or len(positive_treatment_revenue) < 5:
        print(
            "Skipping purchaser revenue model: one variant has fewer than 5 purchasers. "
            "Use a longer window or more traffic."
        )
    else:
        control_revenue_given_purchase = bayesian_bootstrap_mean(
            positive_control_revenue,
            n_samples=N_REVENUE_SAMPLES,
            seed=SEED + 1,
        )
        treatment_revenue_given_purchase = bayesian_bootstrap_mean(
            positive_treatment_revenue,
            n_samples=N_REVENUE_SAMPLES,
            seed=SEED + 2,
        )

        control_revenue_per_user = purchase_samples["control"] * control_revenue_given_purchase
        treatment_revenue_per_user = purchase_samples["treatment"] * treatment_revenue_given_purchase
        revenue_lift = treatment_revenue_per_user - control_revenue_per_user
        revenue_relative_lift = treatment_revenue_per_user / np.maximum(control_revenue_per_user, 1e-12) - 1

        revenue_summary = pd.DataFrame(
            {
                "metric": [
                    "Expected control revenue per user",
                    "Expected treatment revenue per user",
                    "Expected absolute revenue lift",
                    "Expected relative revenue lift",
                    "P(treatment RPU > control RPU)",
                    "94% HDI absolute revenue lift",
                ],
                "value": [
                    money(np.mean(control_revenue_per_user)),
                    money(np.mean(treatment_revenue_per_user)),
                    money(np.mean(revenue_lift)),
                    pct(np.mean(revenue_relative_lift)),
                    pct(np.mean(treatment_revenue_per_user > control_revenue_per_user)),
                    f"{money(hdi_interval(revenue_lift, CREDIBLE_MASS)[0])} to {money(hdi_interval(revenue_lift, CREDIBLE_MASS)[1])}",
                ],
            }
        )
        display(revenue_summary)

        fig, ax = plt.subplots()
        ax.hist(revenue_lift, bins=80, density=True, color="#B279A2", alpha=0.65)
        ax.axvline(0, color="black", linewidth=1.5, label="No revenue lift")
        ax.axvline(np.mean(revenue_lift), color="#E45756", linewidth=2, label="Posterior mean")
        ax.set_title("Posterior Revenue per User Lift")
        ax.set_xlabel("Treatment revenue per user lift")
        ax.set_ylabel("Posterior density")
        ax.xaxis.set_major_formatter(mticker.StrMethodFormatter("${x:,.0f}"))
        ax.legend()
        plt.tight_layout()
        plt.show()

## 11. Prior sensitivity

Priors encode what we believed before seeing the experiment data.

- A weak prior lets data dominate.
- A stronger prior is more conservative.
- With large sample sizes, priors matter less.
- With small sample sizes, priors matter more.

Below we compare three priors:

- `Beta(1, 1)`: flat prior.
- `Beta(2, 8)`: prior mean 20%, useful when conversion is expected to be fairly low.
- `Beta(10, 90)`: prior mean 10%, stronger and more conservative.

In [ ]:
def run_prior_sensitivity(
    df: pd.DataFrame,
    outcome_col: str,
    priors: list[tuple[int, int]],
    n_samples: int = 100_000,
    seed: int = 42,
) -> pd.DataFrame:
    """Compare Beta-Binomial posterior results across prior choices."""
    rows = []
    for prior_alpha, prior_beta in priors:
        samples = bayesian_beta_binomial_ab_test(
            df,
            outcome_col=outcome_col,
            alpha=prior_alpha,
            beta=prior_beta,
            n_samples=n_samples,
            seed=seed,
        )
        rel_low, rel_high = hdi_interval(samples["relative_lift"], CREDIBLE_MASS)
        rows.append(
            {
                "prior": f"Beta({prior_alpha}, {prior_beta})",
                "posterior_mean_control": np.mean(samples["control"]),
                "posterior_mean_treatment": np.mean(samples["treatment"]),
                "prob_treatment_wins": np.mean(samples["treatment"] > samples["control"]),
                "expected_relative_lift": np.mean(samples["relative_lift"]),
                "relative_lift_94_hdi": f"{pct(rel_low)} to {pct(rel_high)}",
            }
        )
    return pd.DataFrame(rows)


prior_sensitivity = run_prior_sensitivity(
    experiment,
    outcome_col="converted_30d",
    priors=[(1, 1), (2, 8), (10, 90)],
    n_samples=N_POSTERIOR_SAMPLES,
    seed=SEED,
)

prior_sensitivity.style.format(
    {
        "posterior_mean_control": "{:.2%}",
        "posterior_mean_treatment": "{:.2%}",
        "prob_treatment_wins": "{:.1%}",
        "expected_relative_lift": "{:.1%}",
    }
)

## 12. Reusable analysis API

The reusable pieces used throughout the notebook are:

- `make_user_level_outcomes(users, orders, revenue_col=None)`
- `summarize_experiment(df, outcome_col)`
- `bayesian_beta_binomial_ab_test(df, outcome_col, variant_col="variant", control_label="control", treatment_label="treatment", alpha=1, beta=1, n_samples=100_000, seed=42)`
- `summarize_posterior_results(samples)`
- `plot_conversion_posteriors(samples)`
- `plot_lift_distribution(samples)`

In a production analytics repo, these can move into a tested package. For an interview or technical blog notebook, keeping them visible makes the assumptions easier to review.

## 13. Final executive summary

Bayesian A/B testing answers decision questions directly:

- How likely is treatment better than control?
- How large is the expected lift?
- How likely is the lift large enough to matter?
- What does the posterior imply for a ship, hold, or continue decision?

For the observed after-the-fact assignment in this notebook, the result should be read as a mechanics walkthrough, not causal product evidence. The correct causal design would assign `variant` before users experience the onboarding or checkout flow, then measure conversion and revenue outcomes after assignment.

In [ ]:
executive_summary = pd.DataFrame(
    {
        "question": [
            "What does the observed data suggest?",
            "Should treatment ship under the illustrative rule?",
            "What does the synthetic effect section show?",
            "What is the main caveat?",
        ],
        "answer": [
            (
                f"Observed treatment conversion is {pct(treatment_rate)} vs control at {pct(control_rate)}; "
                f"posterior P(treatment > control) is {pct(prob_treatment_wins)}."
            ),
            decision,
            "When a small treatment effect is injected, posterior lift and win probability move upward as expected.",
            "This notebook assigns variant after outcomes were generated, so it illustrates Bayesian mechanics rather than proving a real causal treatment effect.",
        ],
    }
)
executive_summary